# Secrets Datasets Preprocessing

### Authors - Shivendra Raghav (1001123853), Amirhosein Morteza (1002187551)
### CSE 5388-002

#### This files reads the SecretBench "Secret Regular Expression.xlsx" files and cleans and creates a JSON with synthetic data. It then reads the common_passwords.csv and adds it to the JSON. The JSON is then saved as "secrets.json" file.

### Import the required libraries

In [209]:
import re
import json
import exrex
import random
import pandas as pd
from profanity_check import predict, predict_prob

### Read the datasets into the dataframes for processing

In [210]:
# Define the filepaths for the datasets
sensitive_data_value_path = "../sensitive_data_values/"
datasets_base_path = sensitive_data_value_path + "datasets/"
secret_regex_filepath = datasets_base_path + "Secret Regular Expression.xlsx"
common_passwords_filepath = datasets_base_path + "common_passwords.csv"

In [211]:
# Load the datasets
secret_regex_df = pd.read_excel(secret_regex_filepath)
common_passwords_df = pd.read_csv(common_passwords_filepath)

In [212]:
# Preprocess the secret regex datasets
secret_regex_df.head()

,Pattern_ID,Secret Type,Regular Expression,Source
0,1,Abbysale API,"(?i)(?:abbysale)(?:.|[\n\r]){0,40}\b([a-z0-9A-...",TruffleHog
1,2,Abstract API,"(?i)(?:abstract)(?:.|[\n\r]){0,40}\b([0-9a-z]{...",TruffleHog
2,3,AbuseIPDB API,"(?i)(?:abuseipdb)(?:.|[\n\r]){0,40}\b([a-z0-9]...",TruffleHog
3,4,AccuWeather API,"(?i)(?:accuweather)(?:.|[\n\r]){0,40}([a-z0-9A...",TruffleHog
4,5,Adafruit API,\b(aio\_[a-zA-Z0-9]{28})\b,TruffleHog


In [213]:
# Preprocess the common passwords datasets
common_passwords_df.head()

,password,length,num_chars,num_digits,num_upper,num_lower,num_special,num_vowels,num_syllables
0,123456,6,0,6,0,0,0,0,1
1,password,8,8,0,0,8,0,2,2
2,12345678,8,0,8,0,0,0,0,1
3,qwerty,6,6,0,0,6,0,1,3
4,123456789,9,0,9,0,0,0,0,1


### Clean the dataframe to remove unwanted columns

In [214]:
# Drop columns that are not needed
secret_regex_df.drop(["Pattern_ID", "Source"], axis=1, inplace=True)

secret_regex_df.head()

,Secret Type,Regular Expression
0,Abbysale API,"(?i)(?:abbysale)(?:.|[\n\r]){0,40}\b([a-z0-9A-..."
1,Abstract API,"(?i)(?:abstract)(?:.|[\n\r]){0,40}\b([0-9a-z]{..."
2,AbuseIPDB API,"(?i)(?:abuseipdb)(?:.|[\n\r]){0,40}\b([a-z0-9]..."
3,AccuWeather API,"(?i)(?:accuweather)(?:.|[\n\r]){0,40}([a-z0-9A..."
4,Adafruit API,\b(aio\_[a-zA-Z0-9]{28})\b


In [215]:
# Drop columns that are not needed
common_passwords_df.drop(["num_chars", "num_digits", "num_upper", "num_lower", "num_special", "num_vowels", "num_syllables"], axis=1, inplace=True)

common_passwords_df.head()

,password,length
0,123456,6
1,password,8
2,12345678,8
3,qwerty,6
4,123456789,9


### Function to assign a generalized secret type

In [216]:
# Fucntion to assign a generalized secret type to the secret
def generalize_secret(secret_type):
    # Convert the secret type to lower case
    secret_type = secret_type.lower()

    # Check if the secret type contains the word "api key"
    if "access token" in secret_type:
        return "Access_Token"
    elif "api token" in secret_type:
        return "API_Token"
    elif "api secret" in secret_type:
        return "API_Secret"
    elif "api key" in secret_type or "api" in secret_type:
        return "API_Key"
    elif "oauth id" in secret_type or "oauth client id" in secret_type:
        return "OAuth_Id"
    elif "oauth secret" in secret_type or "oauth client secret" in secret_type:
        return "OAuth_Secret"
    elif "private key" in secret_type:
        return "Private_Key"
    else:
        return "Other"

### Function to validate the regular expression

In [217]:

# Function to compile the regular expression pattern
def validate_regex(pattern):
    try:
        start_index = pattern.find(r"\b") + 2
        if start_index == -1:
            return None
        end_index = pattern.rfind(r"\b", start_index)
        if end_index == -1:
            return None
        
        # Extract the pattern
        pattern = pattern[start_index:end_index]
        re.compile(pattern)
        return pattern
    except re.error:
        # Return None if the pattern is invalid
        return None

### Create new columns to store the core regular expression and a generalized secret type

In [218]:
# Create a new column to store the core regular expression
secret_regex_df["Valid_Regular_Expression"] = secret_regex_df["Regular Expression"]

# Create a new column to store a generalized Secret Type
secret_regex_df["Secret_Type"] = secret_regex_df["Secret Type"]

### Generalize Secret Type

In [219]:
# Generalize the secret type
secret_regex_df["Secret_Type"] = secret_regex_df["Secret_Type"].apply(generalize_secret)

# Delete the rows with the "Unknown" secret type
secret_regex_df = secret_regex_df[secret_regex_df["Secret_Type"] != "Other"]

secret_regex_df.head()

,Secret Type,Regular Expression,Valid_Regular_Expression,Secret_Type
0,Abbysale API,"(?i)(?:abbysale)(?:.|[\n\r]){0,40}\b([a-z0-9A-...","(?i)(?:abbysale)(?:.|[\n\r]){0,40}\b([a-z0-9A-...",API_Key
1,Abstract API,"(?i)(?:abstract)(?:.|[\n\r]){0,40}\b([0-9a-z]{...","(?i)(?:abstract)(?:.|[\n\r]){0,40}\b([0-9a-z]{...",API_Key
2,AbuseIPDB API,"(?i)(?:abuseipdb)(?:.|[\n\r]){0,40}\b([a-z0-9]...","(?i)(?:abuseipdb)(?:.|[\n\r]){0,40}\b([a-z0-9]...",API_Key
3,AccuWeather API,"(?i)(?:accuweather)(?:.|[\n\r]){0,40}([a-z0-9A...","(?i)(?:accuweather)(?:.|[\n\r]){0,40}([a-z0-9A...",API_Key
4,Adafruit API,\b(aio\_[a-zA-Z0-9]{28})\b,\b(aio\_[a-zA-Z0-9]{28})\b,API_Key


### Validate and remove invalid regexes

In [220]:
# Extract the core regular expression pattern
secret_regex_df["Valid_Regular_Expression"] = secret_regex_df["Valid_Regular_Expression"].apply(validate_regex)

# Delete the rows with None values for Valid_Regular_Expression column
secret_regex_df = secret_regex_df[secret_regex_df["Valid_Regular_Expression"].notnull()]

secret_regex_df.head()

,Secret Type,Regular Expression,Valid_Regular_Expression,Secret_Type
0,Abbysale API,"(?i)(?:abbysale)(?:.|[\n\r]){0,40}\b([a-z0-9A-...",([a-z0-9A-Z]{40}),API_Key
1,Abstract API,"(?i)(?:abstract)(?:.|[\n\r]){0,40}\b([0-9a-z]{...",([0-9a-z]{32}),API_Key
2,AbuseIPDB API,"(?i)(?:abuseipdb)(?:.|[\n\r]){0,40}\b([a-z0-9]...",([a-z0-9]{80}),API_Key
4,Adafruit API,\b(aio\_[a-zA-Z0-9]{28})\b,(aio\_[a-zA-Z0-9]{28}),API_Key
5,Adobe Stock API,"(?i)(?:adobe)(?:.|[\n\r]){0,40}\b([a-z0-9]{32})\b",([a-z0-9]{32}),API_Key


### Check for duplicates

In [221]:
# Check for duplicate regular expression patterns in the Valid_Regular_Expression column
secret_regex_df["Valid_Regular_Expression"].value_counts()

# Drop the duplicate regular expression patterns
secret_regex_df.drop_duplicates(subset="Valid_Regular_Expression", keep="first", inplace=True)

# Drop the Secret Type and Regular Expression columns
secret_regex_df.drop(["Secret Type", "Regular Expression"], axis=1, inplace=True)

### Create a JSON File with variations of the regexes

### Expolatory Data Analysis

In [222]:
secret_regex_df.describe()

,Valid_Regular_Expression,Secret_Type
count,359,359
unique,359,6
top,([a-z0-9A-Z]{40}),API_Key
freq,1,337


### Generrate synthetic secrets from the pattern

In [223]:
# Fucntion to generate strings based on the regex patterns and number of secrets
def generate_synthetic_secret(patterns, num_secrets):
    secrets = []
    for _ in range(num_secrets):
        pattern = random.choice(patterns)
        secret = exrex.getone(pattern)
        secrets.append(secret)
    return secrets

In [224]:
# Get JSON with the Secret_Type as Key and List of Regexes as Value
secret_regex_dict = secret_regex_df.groupby("Secret_Type")["Valid_Regular_Expression"].apply(list).to_dict()

# Generate synthetic secrets for each secret type
synthetic_secrets_dict = {}
for secret_type, regexes in secret_regex_dict.items():
    synthetic_secrets = []
    synthetic_secrets.extend(generate_synthetic_secret(regexes, 1000))
    synthetic_secrets_dict[secret_type] = synthetic_secrets

/var/folders/rz/4djzw0r155g8_kdwpx3gm7y40000gn/T/ipykernel_78266/2647836538.py:8: FutureWarning: Possible nested set at position 2
  synthetic_secrets.extend(generate_synthetic_secret(regexes, 1000))


### Process the Common Passwords Dataset

In [225]:
# Drop rows with passwords length is less than 5
common_passwords_df = common_passwords_df[common_passwords_df["length"] >= 5]

common_passwords_df.head()

,password,length
0,123456,6
1,password,8
2,12345678,8
3,qwerty,6
4,123456789,9


### Check Profanity in the Common Passwords

In [226]:
# Check if the password is profane because the probability of it being used in a work environment is low
def check_profanity(password):
    # Predict the profanity of the passwords
    profanity = predict_prob([password])
    
    # If profane, we return None (we are keeping the level of profanity very low as the library was having a hard time with a lot of passwords)
    if profanity[0] > 0.03:
        return None
    
    return password

### Process Passwords

In [227]:
# Remove duplicate passwords
common_passwords_df.drop_duplicates(subset="password", keep="first", inplace=True)

# Check for profanity in the passwords
common_passwords_df["password"] = common_passwords_df["password"].apply(check_profanity)

# Drop the rows with None values
common_passwords_df = common_passwords_df[common_passwords_df["password"].notnull()]

# Get a random list of common passwords
common_passwords_list = common_passwords_df["password"].tolist()

# Append the list of passwords to the synthetic secrets dictionary
synthetic_secrets_dict["Passwords"] = common_passwords_list

### Save the JSON file

In [228]:
# Save the synthetic secrets in a json file
synthetic_secrets_filepath = sensitive_data_value_path + "secrets.json"
with open(synthetic_secrets_filepath, "w") as f:
    json.dump(synthetic_secrets_dict, f, indent=4)